# Day 4 — Bedrock evaluations, red teaming and CI gating

**Learning objectives.** By the end of this notebook you can:

1. Place offline, nightly and online evaluation on one map and say what each one can and cannot
   catch.
2. Grow a golden set with a "teacher" generator while keeping labels independent of the agent:
   validate each candidate against the data, then use the agent's result only to classify it.
3. Write a red-team case whose assertions hold across paraphrases and would catch the attack
   succeeding.
4. Keep three numbers apart — the product floor, run-to-run noise and the allowed regression —
   and check the decisions they produce with the real gate, `scripts/check_thresholds.py`.
5. **Capstone:** find a seeded regression the aggregate gate lets through, change the gate so it
   rejects that regression while the fixed agent still passes, and write the review a PR needs.
6. *Optional:* build (and, only with explicit consent, submit) an Amazon Bedrock evaluation job,
   call AgentCore Evaluations on a live session, and explain the GitHub OIDC flow.

**Required path:** sections 1–6, offline, no AWS account (lab part 1: sections 1–3; lab part 2:
sections 4–6). Sections 7–9 are optional extensions. Four graded exercises; each check cell
prints `not solved yet` until your code passes, and the *Exercise checklist* cell near the end
lists their status (`STOCKROOM_STRICT_EXERCISES=1` makes an unsolved exercise an error). Every
AWS call is behind an explicit guard and is **never** executed in mock mode.

## Setup

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

# Published repository (docs/DECISIONS.md entry 2). Only used when the package is not already
# installed, e.g. in a fresh Colab or SageMaker Studio kernel.
REPO_URL = "https://github.com/genial-labs-ai/system_agent_harness_aws"
REPO_DIR = Path("stockroom-workshop")

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "1")

if importlib.util.find_spec("stockroom") is None:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    # Pins mirror pyproject.toml; everything else (boto3, pydantic, OpenTelemetry, ...) is
    # resolved from the package metadata by the editable install.
    %pip install -q "deepeval==4.2.8" "ragas==0.4.3" "mcp==2.3.0" "jsonschema==4.26.0" "pyyaml==6.0.3" "langchain-community>=0.3.27,<0.4" "pandas>=2.2" "matplotlib>=3.9" -e ./stockroom-workshop
print("stockroom importable:", importlib.util.find_spec("stockroom") is not None)

In [ ]:
from stockroom.config import detect_mode

config = detect_mode()

In [ ]:
import copy
import datetime as dt
import json
import math
import random
import re
import statistics
from typing import Any

import pandas as pd
import yaml
from IPython.display import Markdown, display

from stockroom.agent.harness import Harness
from stockroom.agent.tools import StockroomData
from stockroom.agent.types import RunResult
from stockroom.config import REPO_ROOT, StockroomConfig
from stockroom.evals.golden import ExpectedTool, GoldenCase, golden_by_id, load_golden
from stockroom.evals.judge import make_judge
from stockroom.evals.metrics import CaseScores, evaluate_case
from stockroom.evals.report import results_document
from stockroom.evals.stats import floor_is_safe, required_max_drop, run_to_run
from stockroom.exercises import exercise_passed, exercise_pending, exercise_summary

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)
cases = load_golden()
by_id = golden_by_id()
golden_queries = {c.query for c in cases}
judge = make_judge(config)
JUDGE_LABEL = f"{judge.name}:{judge.rubric_version}"
harness = Harness(config)
data = StockroomData.load(config.data_dir)
SCRIPTS_DIR = REPO_ROOT / "scripts"
print("repository root:", REPO_ROOT)

## 1. Offline, nightly, online

| | what runs | when | catches | cannot catch |
|---|---|---|---|---|
| **offline** (`make ci`) | golden set + red team through the mock harness | every PR | regressions in tool selection, arguments, termination, guards, injection handling | real-model behaviour, drift in the live model |
| **nightly** (`agent_eval_nightly.yml`) | the same suites against Bedrock, repeated to measure run-to-run spread | on demand (`workflow_dispatch`, no schedule), report only | model-side regressions, judge disagreement, cost | traffic you did not write a case for |
| **online** (AgentCore Evaluations, CloudWatch) | evaluators over real session traces | continuously | new query types, user-visible failures | nothing is deterministic; needs sampling and human review |

The rest of this notebook moves left to right along that table.

## 2. Growing the golden set: labels from the data, not from the agent

Golden sets need to grow. A *teacher* proposes new queries; the **labels still come from the
data files**, never from the agent under test. In mock mode the teacher is template-based; in live
mode it asks the judge model (a different family from the agent) to paraphrase the templates
through the Converse API — the expected facts are still looked up in `data/`.

The tempting filter — "keep a candidate when the agent passes it" — is wrong twice over. It
throws away exactly the valid hard cases a suite exists to hold, and it keeps a wrong label
whenever the agent happens to agree with it. So the order is: **validate the label against the
data first, then run the agent to classify the accepted case** (passing, failing, or
needs investigation).

In [ ]:
STOCK_TEMPLATES = [
    "How many units of {sku} do we have on hand?",
    "Could you check the stock position for {sku}?",
]
ORDER_TEMPLATES = [
    "Can you give me an update on order {order_id}?",
    "What is happening with order {order_id}?",
]


def stock_case(case_id: str, product: dict[str, Any], query: str) -> GoldenCase:
    level = product["stock_level"]
    fact = "out of stock" if level == 0 else str(level)
    return GoldenCase(
        id=case_id,
        category="stock_lookup",
        query=query,
        expected_tools=[ExpectedTool(name="get_stock_level", args={"sku": product["sku"]})],
        expected_facts=[fact],
        reference_answer=f"{product['sku']} ({product['name']}) has {level} units in stock.",
        tags=["synthetic"],
    )


def order_case(case_id: str, order: dict[str, Any], query: str) -> GoldenCase:
    return GoldenCase(
        id=case_id,
        category="order_status",
        query=query,
        expected_tools=[ExpectedTool(name="get_order_status", args={"order_id": order["order_id"]})],
        expected_facts=[order["status"]],
        reference_answer=f"Order {order['order_id']} is {order['status']}.",
        tags=["synthetic"],
    )


def template_teacher(seed: int = 7) -> list[GoldenCase]:
    """Deterministic candidate generator over the catalogue and the order book."""
    rng = random.Random(seed)
    products = rng.sample(sorted(data.catalogue, key=lambda p: p["sku"]), 5)
    orders = rng.sample(sorted(data.orders, key=lambda o: o["order_id"]), 4)
    out: list[GoldenCase] = []
    for i, p in enumerate(products):
        out.append(stock_case(f"S{i + 1:03d}", p, rng.choice(STOCK_TEMPLATES).format(sku=p["sku"])))
    for i, o in enumerate(orders, start=len(out)):
        query = rng.choice(ORDER_TEMPLATES).format(order_id=o["order_id"])
        out.append(order_case(f"S{i + 1:03d}", o, query))
    # Four more, of the kinds teachers and tired humans really produce:
    # S010 copies a golden case outright.
    out.append(by_id["G007"].model_copy(update={"id": "S010", "tags": ["synthetic"]}))
    # S011 has a wrong stock level (the agent will disagree with it).
    s011 = stock_case("S011", data.by_sku["SKU-1003"], "How much SKU-1003 stock is left?")
    out.append(s011.model_copy(update={"expected_facts": [str(data.by_sku["SKU-1003"]["stock_level"] + 7)]}))
    # S012 copied the reorder point into the label (the agent's answer happens to contain it).
    s012 = stock_case("S012", data.by_sku["SKU-1006"], "How many units of SKU-1006 do we have on hand?")
    out.append(s012.model_copy(update={"expected_facts": [str(data.by_sku["SKU-1006"]["reorder_point"])]}))
    # S013 is a perfectly valid question, typed the way people type SKUs.
    out.append(stock_case("S013", data.by_sku["SKU-1002"], "How many units of SKU 1002 are left?"))
    return out


def bedrock_teacher(cfg: StockroomConfig, seed: int = 7) -> list[GoldenCase]:
    """Live only: ask the judge model to paraphrase the template queries (labels stay data-derived)."""
    from stockroom.agent.bedrock_adapter import make_bedrock_runtime_client
    from stockroom.evals.judge import parse_json_object

    _, teacher_model = cfg.require_live_models()
    client = make_bedrock_runtime_client(cfg)
    rephrased: list[GoldenCase] = []
    for case in template_teacher(seed):
        prompt = (
            "Rephrase the following warehouse support question in one natural sentence. Keep every "
            f"identifier exactly as written. Reply with JSON {{\"query\": \"...\"}} only.\n\n{case.query}"
        )
        resp = client.converse(
            modelId=teacher_model,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 200, "temperature": 0.7},
        )
        text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
        new_query = str(parse_json_object(text).get("query") or case.query)
        rephrased.append(case.model_copy(update={"query": new_query}))
    return rephrased


candidates = bedrock_teacher(config) if config.is_live else template_teacher()
print(f"{len(candidates)} candidates from the {'Bedrock' if config.is_live else 'template'} teacher")
pd.DataFrame([{"id": c.id, "query": c.query, "expected_tools": [(t.name, t.args) for t in c.expected_tools], "expected_facts": c.expected_facts} for c in candidates])

Three helpers for the validation rules. `data_facts()` is the oracle: what the data says the
answer must contain for this tool call. `signature()` is what a case *exercises* (its category and
expected tool calls); two cases with the same signature walk the same trajectory, so the second
adds little coverage even when it is worded differently. `normalise()` lets "SKU 1002" match
`SKU-1002`.

In [ ]:
def data_facts(case: GoldenCase) -> set[str] | None:
    """The facts the data requires for this case's single lookup, or None when it cannot be checked."""
    if len(case.expected_tools) != 1:
        return None
    call = case.expected_tools[0]
    if call.name == "get_stock_level" and call.args.get("sku") in data.by_sku:
        level = data.by_sku[call.args["sku"]]["stock_level"]
        return {"out of stock" if level == 0 else str(level)}
    if call.name == "get_order_status" and call.args.get("order_id") in data.by_order:
        return {data.by_order[call.args["order_id"]]["status"]}
    return None


def signature(case: GoldenCase) -> tuple[str, tuple[tuple[str, str], ...]]:
    return case.category, tuple((t.name, json.dumps(t.args, sort_keys=True)) for t in case.expected_tools)


def normalise(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())


golden_signatures = {signature(c): c.id for c in cases}
print("example signature:", signature(by_id["G019"]), "->", golden_signatures[signature(by_id["G019"])])

### Exercise 1 — validate candidates without asking the agent

Write `validate_candidate(case)`, returning the list of reasons to reject the case (an empty list
means it is valid). It must not look at any agent run. Reject a candidate when:

1. it has no expected facts;
2. an identifier in its expected tool arguments cannot be found in the query (compare with
   `normalise()`);
3. its expected facts are not exactly what `data_facts()` says (or the data cannot check them);
4. its query is already in the golden set (`golden_queries`);
5. its `signature()` matches a golden case (a near-duplicate, however it is worded).

Success criteria: exactly `S007`, `S010`, `S011` and `S012` are rejected; the valid hard case
`S013` is accepted even though the agent fails it, and `S012` is rejected even though the agent
passes it.

In [ ]:
def validate_candidate(case: GoldenCase) -> list[str]:
    """Reasons to reject ``case`` as a golden case; [] means the label is valid."""
    reasons: list[str] = []
    # TODO: implement the five rules above with data_facts(), signature(), normalise(),
    #       golden_queries and golden_signatures.
    return reasons

The agent runs *after* validation and never decides acceptance. For the table below it runs on
every candidate so you can see what agent-based filtering would have done.

In [ ]:
candidate_runs: dict[str, tuple[RunResult, CaseScores]] = {}
for c in candidates:
    r = harness.run(c.query, case_id=c.id)
    candidate_runs[c.id] = (r, evaluate_case(c, r, judge))


def classify(scores: CaseScores) -> str:
    """What an accepted case means for the agent: the agent never decides whether the case is valid."""
    if scores.judge_passed is not None and scores.judge_passed != scores.answer_correctness_deterministic:
        return "investigate: judge and fact check disagree"
    if scores.judge_passed and scores.tool_selection == 1.0 and scores.termination_match:
        return "passing"
    return "failing: keep it, add it to the error-analysis queue"


validation = {c.id: validate_candidate(c) for c in candidates}
pd.DataFrame(
    [
        {
            "id": c.id,
            "valid label": not validation[c.id],
            "reasons": "; ".join(validation[c.id]),
            "agent passed": bool(candidate_runs[c.id][1].judge_passed),
            "tools": candidate_runs[c.id][1].tool_names,
            "classification": classify(candidate_runs[c.id][1]) if not validation[c.id] else "—",
        }
        for c in candidates
    ]
)

In [ ]:
rejected = sorted(cid for cid, reasons in validation.items() if reasons)
if not rejected:
    exercise_pending("day4.ex1", "every candidate is accepted")
else:
    assert rejected == ["S007", "S010", "S011", "S012"], f"rejected {rejected}"
    assert candidate_runs["S012"][1].judge_passed, "S012 is the case the agent agrees with"
    assert not candidate_runs["S013"][1].judge_passed, "S013 is the valid case the agent fails"
    exercise_passed(
        "day4.ex1",
        f"accepted {len(candidates) - len(rejected)} (S013 as a known failure), rejected {rejected}",
    )

## 3. The Promptfoo red-team suite, in-process

`promptfooconfig.yaml` holds ten golden cases and eight hand-written red-team cases; every
red-team case encodes *"the attack must not succeed"*. In CI, `make promptfoo` runs it with the
pinned Promptfoo CLI through `scripts/promptfoo_provider.py`. Here we execute the same test
definitions and the same Python assertion functions (`scripts/promptfoo_asserts.py`) directly,
so you can see exactly what each assertion checks.

In [ ]:
import importlib.util as _ilu


def load_script(name: str):
    path = SCRIPTS_DIR / f"{name}.py"
    spec = _ilu.spec_from_file_location(name, path)
    assert spec is not None and spec.loader is not None, f"cannot load {path}"
    module = _ilu.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


asserts = load_script("promptfoo_asserts")
gate = load_script("check_thresholds")
promptfoo_cfg = yaml.safe_load((REPO_ROOT / "promptfooconfig.yaml").read_text(encoding="utf-8"))
redteam_tests = [t for t in promptfoo_cfg["tests"] if t["metadata"]["suite"] == "redteam"]
print(f"{len(redteam_tests)} red-team tests; assertion functions: "
      f"{[n for n in dir(asserts) if not n.startswith('_')]}")


def provider_output(run: RunResult) -> str:
    """The JSON envelope scripts/promptfoo_provider.py returns to Promptfoo."""
    return json.dumps(
        {
            "answer": run.final_answer,
            "tools": run.tool_names,
            "termination": run.termination_reason.value,
            "quarantined": sum(r.quarantined_lines for r in run.tool_records),
            "steps": run.usage.model_calls,
        },
        ensure_ascii=False,
    )


def run_assertion(assertion: dict[str, Any], output: str) -> dict[str, Any]:
    if assertion["type"] == "is-json":
        json.loads(output)
        return {"pass": True, "reason": "valid JSON"}
    if assertion["type"] == "python":
        fn_name = str(assertion["value"]).rsplit(":", 1)[-1]
        fn = getattr(asserts, fn_name)
        return fn(output, {"config": assertion.get("config", {})})
    raise ValueError(f"unsupported assertion type {assertion['type']}")


def run_redteam(cfg: StockroomConfig) -> pd.DataFrame:
    h = Harness(cfg)
    rows = []
    for test in redteam_tests:
        out = provider_output(h.run(test["vars"]["query"]))
        verdicts = [
            run_assertion(a, out) for a in [*promptfoo_cfg["defaultTest"]["assert"], *test["assert"]]
        ]
        failed = [v["reason"] for v in verdicts if not v["pass"]]
        rows.append(
            {
                "case_id": test["metadata"]["case_id"],
                "category": test["metadata"]["category"],
                "passed": not failed,
                "failed_assertions": failed,
            }
        )
    return pd.DataFrame(rows)


print("fixed configuration")
redteam_clean = run_redteam(config)
display(redteam_clean)
print("with injection_unguarded")
redteam_weak = run_redteam(config.replace(weaknesses="injection_unguarded"))
display(redteam_weak)

In [ ]:
def promptfoo_summary_from(df: pd.DataFrame) -> dict[str, Any]:
    """Same shape as check_thresholds.promptfoo_summary() builds from reports/promptfoo_results.json."""
    failed = df[~df["passed"]]
    return {
        "available": True,
        "suites": {
            "redteam": {"total": len(df), "passed": int(df["passed"].sum()), "failed": len(failed)}
        },
        "failures": [
            {"suite": "redteam", "case_id": r.case_id, "reason": "; ".join(r.failed_assertions)}
            for r in failed.itertuples()
        ],
        "seen": list(df["case_id"]),
    }


promptfoo_clean = promptfoo_summary_from(redteam_clean)
promptfoo_weak = promptfoo_summary_from(redteam_weak)
print(promptfoo_clean["suites"], "|", promptfoo_weak["suites"])

### Exercise 2 — a red-team case that would catch the attack

Write one more red-team case in the Promptfoo shape: a `query`, at least one paraphrase under
`variants`, and a list of `(assertion function, config)` pairs from `scripts/promptfoo_asserts.py`.
The case passes when the attack **fails**. Ideas: customer details through an order, an action
the agent has no tool for, an instruction smuggled into a legitimate question.

Success criteria: the queries are new; there is at least one **trajectory** assertion
(`expect_tools` or `no_tool_called`; a polite refusal after the forbidden tool ran is not safe);
every assertion passes on the fixed agent for the query *and* every variant; and at least one
assertion fails on `unsafe_output()` — the envelope of a run in which the attack succeeded. An
assertion set that passes there too cannot catch anything.

In [ ]:
TRAJECTORY_ASSERTS = {"expect_tools", "no_tool_called"}
TOOL_NAMES = sorted(harness.specs)


def unsafe_output(case: dict[str, Any]) -> str:
    """A run where the attack worked: every tool executed and the answer says what the case forbids."""
    forbidden = [
        value
        for name, cfg in case["asserts"]
        if name == "answer_not_contains"
        for value in cfg.get("values", [])
    ]
    return json.dumps(
        {
            "answer": " ".join(forbidden) or "Done, as you asked.",
            "tools": TOOL_NAMES,
            "termination": "COMPLETED",
            "quarantined": 0,
            "steps": 2,
        }
    )

In [ ]:
NEW_RED_TEAM_CASE: dict[str, Any] | None = None
# TODO: NEW_RED_TEAM_CASE = {
#     "case_id": "RT09", "category": "...", "query": "...", "variants": ["..."],
#     "asserts": [("no_tool_called", {"tool": "create_restock_request"}), ...],
# }

In [ ]:
existing_queries = {t["vars"]["query"] for t in promptfoo_cfg["tests"]}
if NEW_RED_TEAM_CASE is None:
    exercise_pending("day4.ex2")
else:
    queries = [NEW_RED_TEAM_CASE["query"], *NEW_RED_TEAM_CASE.get("variants", [])]
    assert len(queries) >= 2, "add at least one paraphrase under 'variants'"
    assert not set(queries) & existing_queries, "a query is already in the suite"
    names = {name for name, _ in NEW_RED_TEAM_CASE["asserts"]}
    assert names & TRAJECTORY_ASSERTS, f"add a trajectory assertion: one of {sorted(TRAJECTORY_ASSERTS)}"
    for q in queries:
        out = provider_output(harness.run(q))
        failed = {
            name: getattr(asserts, name)(out, {"config": cfg})["reason"]
            for name, cfg in NEW_RED_TEAM_CASE["asserts"]
            if not getattr(asserts, name)(out, {"config": cfg})["pass"]
        }
        assert not failed, f"{q!r}: the attack succeeded or the assertion is wrong: {failed}"
    unsafe = unsafe_output(NEW_RED_TEAM_CASE)
    caught = [
        name
        for name, cfg in NEW_RED_TEAM_CASE["asserts"]
        if not getattr(asserts, name)(unsafe, {"config": cfg})["pass"]
    ]
    assert caught, "every assertion also passes when the attack succeeds"
    exercise_passed(
        "day4.ex2",
        f"{NEW_RED_TEAM_CASE['case_id']} holds on {len(queries)} phrasings; on a successful attack "
        f"{caught} fail",
    )

## 4. Three numbers, kept apart: floor, noise, allowed regression

`eval_thresholds.yaml` mixes three different questions, and each has its own answer:

| question | where it lives | where the number comes from |
|---|---|---|
| What is the worst quality we will merge? | `gates.*.min` | a **product decision** (the brief's 0.85); the data only checks that `main` clears it on a bad run: mean − 2·sd ≥ floor |
| How much does the metric move when nothing changed? | measured, `run_to_run` in `reports/eval_results.json` | rerun the same suite on the same code; sd of the per-repeat values |
| How big a drop versus the baseline fails a PR? | `regression.max_drop_vs_baseline` | must exceed noise: a PR run and the baseline run differ by noise with sd·√2, so ≥ 2·√2·sd |

`stockroom.evals.stats` implements these (`run_to_run()`, `floor_is_safe()`,
`required_max_drop()`), the regression suite writes `run_to_run` into the results file, and the
gate's summary warns when a configured threshold sits inside the measured noise. First, the mock
suite three times:

In [ ]:
REPEATS = 3
_suite_cache: dict[tuple[str, int], dict[str, Any]] = {}


def results_for(cfg: StockroomConfig, repeats: int = 1) -> dict[str, Any]:
    """The eval_results.json document for ``cfg``; each configuration is scored once and cached."""
    key = (",".join(sorted(cfg.weaknesses)), repeats)
    if key not in _suite_cache:
        h = Harness(cfg)
        scored = [
            (r, evaluate_case(c, h.run(c.query, case_id=c.id), judge))
            for r in range(repeats)
            for c in cases
        ]
        _suite_cache[key] = results_document(cfg, scored, judge_label=JUDGE_LABEL)
    return _suite_cache[key]


mock_repeats = results_for(config, REPEATS)
display(pd.DataFrame(mock_repeats["run_to_run"]).T)

Every repeat is identical and the spread is exactly zero: the mock simulator is deterministic.
That demonstrates **reproducibility, not reliability** — it is why the PR gate can use the floors
directly, and why the numbers that matter for thresholds come from live runs.

Below is an **illustrative** set of five nightly live runs. These values are synthetic, made up
for the exercise and not measured; the procedure is what carries over.

In [ ]:
ILLUSTRATIVE_NIGHTLY = {  # synthetic teaching data, not a measurement
    "tool_selection_accuracy": [0.90, 0.88, 0.92, 0.89, 0.91],
    "answer_correctness": [0.86, 0.90, 0.84, 0.88, 0.87],
}
thresholds = yaml.safe_load((REPO_ROOT / "eval_thresholds.yaml").read_text(encoding="utf-8"))
FLOOR = thresholds["gates"]["answer_correctness"]["min"]
print("product floor for both metrics:", FLOOR, "| shipped max_drop_vs_baseline:",
      thresholds["regression"]["max_drop_vs_baseline"])

Decisions on fixed examples, through the real `gate.evaluate()`. The baseline is the illustrative
nightly mean of `answer_correctness` (0.87); three PR runs are compared with it under a given
`max_drop_vs_baseline` — the regression rule on its own, so the floor does not interfere.

In [ ]:
PROVENANCE = {k: mock_repeats[k] for k in ("mode", "agent_model_id", "judge", "dataset")}
EXAMPLES = {"same as baseline": 0.87, "within noise": 0.81, "real regression": 0.78}


def regression_decisions(max_drop: float) -> dict[str, str]:
    rule = {"regression": {"max_drop_vs_baseline": max_drop, "metrics": ["answer_correctness"]}}
    base = {**PROVENANCE, "metrics": {"answer_correctness": 0.87}}
    out = {}
    for label, value in EXAMPLES.items():
        pr = {**PROVENANCE, "metrics": {"answer_correctness": value}, "cases": []}
        _rows, failures = gate.evaluate(rule, pr, gate.promptfoo_summary(None), base)
        out[label] = "fail" if failures else "pass"
    return out


print("with the shipped max_drop 0.05:", regression_decisions(0.05))

### Exercise 3 — thresholds from run-to-run spread

Implement `propose(per_run, floor)` for one metric's per-run values. Return a dictionary with

* `sd` — the sample standard deviation of the runs (`statistics.stdev`);
* `floor_ok` — whether mean − 2·sd ≥ floor;
* `max_drop` — 2·√2·sd rounded **up** to two decimals.

Success criteria: your numbers equal `stockroom.evals.stats` on both metrics; `answer_correctness`
fails the floor check (0.85 sits inside its noise) while `tool_selection_accuracy` passes it; and
with the larger of your two `max_drop` values the gate passes the "within noise" PR, fails the real
regression, and passes the unchanged run — where the shipped 0.05 would have failed the noisy one.

In [ ]:
def propose(per_run: list[float], floor: float) -> dict[str, Any] | None:
    """{"sd": ..., "floor_ok": ..., "max_drop": ...} for one metric (None = not solved yet)."""
    # TODO: statistics.fmean / statistics.stdev, math.sqrt and math.ceil (two decimals, rounded up).
    return None

In [ ]:
proposals = {m: propose(v, FLOOR) for m, v in ILLUSTRATIVE_NIGHTLY.items()}
if any(p is None for p in proposals.values()):
    exercise_pending("day4.ex3")
else:
    for metric, values in ILLUSTRATIVE_NIGHTLY.items():
        ref = run_to_run(values)
        got = proposals[metric]
        assert abs(got["sd"] - ref["sd"]) < 1e-3, f"{metric}: sd {got['sd']:.4f}, expected {ref['sd']}"
        assert got["floor_ok"] == floor_is_safe(ref["mean"], ref["sd"], FLOOR), f"{metric}: floor check"
        assert got["max_drop"] == required_max_drop(ref["sd"]), f"{metric}: max_drop {got['max_drop']}"
    assert not proposals["answer_correctness"]["floor_ok"] and proposals["tool_selection_accuracy"]["floor_ok"]
    max_drop = max(p["max_drop"] for p in proposals.values())
    decisions = regression_decisions(max_drop)
    assert decisions == {"same as baseline": "pass", "within noise": "pass", "real regression": "fail"}, decisions
    assert regression_decisions(0.05)["within noise"] == "fail"
    exercise_passed(
        "day4.ex3",
        f"max_drop {max_drop} -> {decisions}; floor safe: "
        f"{ {m: p['floor_ok'] for m, p in proposals.items()} }",
    )

The gate reaches the same verdicts on its own. Give it a results document whose `run_to_run`
block holds the illustrative runs and it prints the warnings that appear in a live run's
`reports/summary.md`:

In [ ]:
illustrative_results = {
    **mock_repeats,
    "run_to_run": {m: run_to_run(v) for m, v in ILLUSTRATIVE_NIGHTLY.items()},
}
print("\n".join(gate.noise_notes(thresholds, illustrative_results)))

## 5. The gate on today's suite, and what it cannot see

CI calls `scripts/check_thresholds.py` on `reports/eval_results.json`; its pure functions —
`check_evidence()`, `evaluate()` and `render()` — work on dictionaries, so we can feed them the
in-notebook results. The committed baseline (`reports/baseline/main.json`, written by
`make baseline`) records its provenance and per-case outcomes, so the gate can refuse an
incompatible baseline and list the cases that changed.

Start with the gate as it stood before today: **aggregate floors plus the regression rule**.

In [ ]:
manifest_path = config.data_dir / "golden" / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
baseline = json.loads((REPO_ROOT / "reports" / "baseline" / "main.json").read_text())
AGGREGATE_ONLY = {k: v for k, v in thresholds.items() if k not in ("category_gates", "cost")}
FLAGS = ["ambiguous_tool_desc", "naive_retry", "oversized_payload", "injection_unguarded"]
suites = {"fixed": results_for(config)} | {f: results_for(config.replace(weaknesses=f)) for f in FLAGS}
print("evidence problems on the fixed run:", gate.check_evidence(suites["fixed"], manifest, manifest_path))


def verdicts(th: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for name, doc in suites.items():
        _rows, failures = gate.evaluate(th, doc, promptfoo_clean, baseline)
        rows.append({"configuration": name, "gate": "FAIL" if failures else "pass", "failures": failures})
    return pd.DataFrame(rows)


verdicts(AGGREGATE_ONLY)

Two seeded regressions sail through. Read the summary for `naive_retry`: every aggregate stays
above its floor and within 0.05 of the baseline, yet two cases now loop until `MAX_STEPS` and the
mean input tokens rose by 15 %. The "Cases that changed vs baseline" list — a paired comparison of
each case with its own baseline outcome — shows exactly where the damage is.

In [ ]:
_rows, _failures = gate.evaluate(AGGREGATE_ONLY, suites["naive_retry"], promptfoo_clean, baseline)
display(Markdown(gate.render(suites["naive_retry"], _rows, _failures, promptfoo_clean, baseline, AGGREGATE_ONLY)))

## 6. Capstone: make the gate reject what it missed

This is the deliverable for the afternoon. You are reviewing a PR that "simplified retries"
(`naive_retry`) and another that "returns full product records" (`oversized_payload`). Both pass
the aggregate gate above. Make an **evaluation change** — in the thresholds; a new golden case or
a Day 3 trajectory assertion are good additions to discuss in your review — so that:

* the fixed agent still passes (a gate that fails `main` gets disabled);
* `naive_retry` and `oversized_payload` fail, and `ambiguous_tool_desc` and `injection_unguarded`
  still fail;
* the change is justified in a review note (at least 60 words) that says which evidence you used
  and what it means for the committed baseline (does it need regenerating? why or why not?).

The check cell saves the before/after gate summaries and your review under `reports/day4/`, so you
can attach them to the PR discussion in the review block.

In [ ]:
CAPSTONE_THRESHOLDS: dict[str, Any] | None = None
CAPSTONE_REVIEW = ""
# TODO: CAPSTONE_THRESHOLDS = copy.deepcopy(AGGREGATE_ONLY), then add the rule(s) the evidence
#       calls for (evaluate() also reads `category_gates` and `cost` blocks; see
#       scripts/check_thresholds.py). Write CAPSTONE_REVIEW as the PR review you would post.

In [ ]:
if CAPSTONE_THRESHOLDS is None:
    exercise_pending("day4.ex4")
else:
    table = verdicts(CAPSTONE_THRESHOLDS).set_index("configuration")
    assert table.loc["fixed", "gate"] == "pass", f"the fixed agent fails: {table.loc['fixed', 'failures']}"
    still_passing = [f for f in FLAGS if table.loc[f, "gate"] == "pass"]
    assert not still_passing, f"still passes the gate: {still_passing}"
    words = len(CAPSTONE_REVIEW.split())
    assert words >= 60, f"the review has {words} words; explain the evidence and the baseline"
    assert "baseline" in CAPSTONE_REVIEW.lower(), "say what the change means for the baseline"
    out_dir = REPO_ROOT / "reports" / "day4"
    out_dir.mkdir(parents=True, exist_ok=True)
    for label, th in (("before", AGGREGATE_ONLY), ("after", CAPSTONE_THRESHOLDS)):
        rows, failures = gate.evaluate(th, suites["naive_retry"], promptfoo_clean, baseline)
        summary = gate.render(suites["naive_retry"], rows, failures, promptfoo_clean, baseline, th)
        (out_dir / f"capstone_naive_retry_{label}.md").write_text(summary, encoding="utf-8")
    (out_dir / "capstone_thresholds.yaml").write_text(yaml.safe_dump(CAPSTONE_THRESHOLDS, sort_keys=False))
    (out_dir / "capstone_review.md").write_text(CAPSTONE_REVIEW.strip() + "\n", encoding="utf-8")
    display(table)
    exercise_passed(
        "day4.ex4",
        f"fixed passes, {len(FLAGS)} seeded regressions fail; evidence in {out_dir.relative_to(REPO_ROOT)}",
    )

Compare your change with what the repository ships. `eval_thresholds.yaml` (version 2) carries a
per-category termination invariant and a token-cost limit, and
`tests/test_seeded_weaknesses.py` asserts that the shipped gate rejects every weakness flag on
its own while the fixed agent passes:

In [ ]:
display(verdicts(thresholds))
_rows, _failures = gate.evaluate(thresholds, suites["fixed"], promptfoo_clean, baseline)
display(Markdown(gate.render(suites["fixed"], _rows, _failures, promptfoo_clean, baseline, thresholds)))

## 7. Optional extension (≈30 min): Amazon Bedrock model evaluation with your own responses

Building and validating the payload runs offline; only the submission cell needs an AWS account,
and it refuses to run without explicit consent.

Bedrock's `CreateEvaluationJob` API runs LLM-as-a-judge evaluations over a JSONL prompt dataset
in S3. With a *precomputed inference source* the responses are supplied in the dataset instead of
being generated by Bedrock, which is what we need for an agent: each line carries the prompt, the
reference answer and the agent's response (`prompt`, `referenceResponse`, `category`,
`modelResponses[{response, modelIdentifier}]`), and the built-in judge metrics score them.

* API reference: https://docs.aws.amazon.com/bedrock/latest/APIReference/API_CreateEvaluationJob.html
* User guide: https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-jobs-management-create.html

The cell below builds the dataset from the golden set and the agent's own answers, builds the
exact request payload and validates it **offline** against the Bedrock service model that ships
with botocore (so the shape is checked without any network call). Submission is a separate cell
with three explicit guards. Note the evaluation job needs its own service role (see
`docs/aws/README.md`); it is not part of the CI role on purpose.

In [ ]:
import tempfile

from botocore import session as botocore_session
from botocore.validate import validate_parameters

SOURCE_ID = "stockroom-harness-" + (config.agent_model_id or "mock").replace(".", "-").replace(":", "-")[:40]
dataset_version = str(manifest["version"]).replace(".", "-")
S3_BUCKET = config.s3_bucket or "<S3_BUCKET>"
S3_PREFIX = config.s3_prefix
JUDGE_MODEL_ID = config.judge_model_id or "<JUDGE_MODEL_ID>"  # from the environment in live mode
EVAL_ROLE_ARN = os.environ.get("EVAL_ROLE_ARN", "<EVAL_ROLE_ARN>")

byoi_rows = []
for case in cases:
    run = harness.run(case.query, case_id=case.id)
    byoi_rows.append(
        {
            "prompt": case.query,
            "referenceResponse": case.reference_answer,
            "category": case.category,
            "modelResponses": [{"response": run.final_answer, "modelIdentifier": SOURCE_ID}],
        }
    )
byoi_dir = Path(tempfile.mkdtemp(prefix="stockroom-byoi-"))
byoi_file = byoi_dir / f"golden_byoi_{dataset_version}.jsonl"
byoi_file.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in byoi_rows) + "\n")
dataset_s3_uri = f"s3://{S3_BUCKET}/{S3_PREFIX}/evaluations/{dataset_version}/{byoi_file.name}"
print(f"{len(byoi_rows)} lines written to {byoi_file}\n")
print(json.dumps(byoi_rows[0], indent=2)[:600])

In [ ]:
job_payload: dict[str, Any] = {
    "jobName": f"stockroom-golden-{dataset_version}-{dt.datetime.now(dt.UTC):%Y%m%d%H%M}",
    "jobDescription": "Stockroom agent answers over the golden set, graded by an LLM judge",
    "roleArn": EVAL_ROLE_ARN,
    "applicationType": "ModelEvaluation",
    "evaluationConfig": {
        "automated": {
            "datasetMetricConfigs": [
                {
                    "taskType": "QuestionAndAnswer",
                    "dataset": {
                        "name": f"stockroom-golden-{dataset_version}",
                        "datasetLocation": {"s3Uri": dataset_s3_uri},
                    },
                    "metricNames": [
                        "Builtin.Correctness",
                        "Builtin.Completeness",
                        "Builtin.Faithfulness",
                        "Builtin.Helpfulness",
                    ],
                }
            ],
            "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": JUDGE_MODEL_ID}]},
        }
    },
    "inferenceConfig": {
        "models": [{"precomputedInferenceSource": {"inferenceSourceIdentifier": SOURCE_ID}}]
    },
    "outputDataConfig": {
        "s3Uri": f"s3://{S3_BUCKET}/{S3_PREFIX}/evaluations/{dataset_version}/output/"
    },
}
operation = botocore_session.get_session().get_service_model("bedrock").operation_model(
    "CreateEvaluationJob"
)
validate_parameters(job_payload, operation.input_shape)  # raises ParamValidationError on a bad shape
print("payload matches the CreateEvaluationJob input shape (offline check)\n")
print(json.dumps(job_payload, indent=2))

### Submitting the job (live only, explicit consent)

The job is submitted **only** when all three hold: `STOCKROOM_MODE=live`,
`STOCKROOM_CONFIRM_AWS_SPEND=1` and `EVAL_ROLE_ARN` is set (plus `S3_BUCKET` for the dataset).
Anything else prints what is missing and stops. Submitting incurs Bedrock charges for the judge
model.

In [ ]:
missing = [
    name
    for name, ok in (
        ("STOCKROOM_MODE=live", config.is_live),
        ("STOCKROOM_CONFIRM_AWS_SPEND=1", config.confirm_aws_spend),
        ("EVAL_ROLE_ARN", bool(os.environ.get("EVAL_ROLE_ARN"))),
        ("S3_BUCKET", bool(config.s3_bucket)),
    )
    if not ok
]
if missing:
    print("Not submitting the evaluation job; missing:", ", ".join(missing))
else:
    import boto3

    key = dataset_s3_uri.split(f"s3://{S3_BUCKET}/", 1)[1]
    boto3.client("s3", region_name=config.aws_region).upload_file(str(byoi_file), S3_BUCKET, key)
    bedrock = boto3.client("bedrock", region_name=config.aws_region)
    response = bedrock.create_evaluation_job(**job_payload)
    print("submitted:", response["jobArn"])
    print("poll with bedrock.get_evaluation_job(jobIdentifier=jobArn); results land in outputDataConfig.s3Uri")

## 8. Optional extension (live only): AgentCore Evaluations on a session

For agents deployed on AgentCore Runtime, **AgentCore Evaluations** scores session traces with
built-in evaluators (`Builtin.Helpfulness`, `Builtin.ToolSelectionAccuracy`, ...). The on-demand
flow documented at
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-on-demand.html is:

1. the agent runs on AgentCore Runtime instrumented with ADOT, with CloudWatch Transaction Search
   enabled;
2. download the session's spans from CloudWatch Logs with a Logs Insights query;
3. call `bedrock-agentcore.evaluate(evaluatorId=..., evaluationInput={"sessionSpans": [...]})`.

Our mock harness does not run on AgentCore Runtime, so this cell runs only with the same explicit
consent as the Bedrock job (`STOCKROOM_MODE=live`, `STOCKROOM_CONFIRM_AWS_SPEND=1`) and an
`AGENTCORE_SESSION_ID`; Logs Insights queries and evaluations are billed. The polling loop has a
deadline, stops on every terminal status the Logs API defines (`Complete`, `Failed`,
`Cancelled`, `Timeout`, `Unknown`) and cancels its query when the deadline passes.

In [ ]:
import time

AGENTCORE_SESSION_ID = os.environ.get("AGENTCORE_SESSION_ID")
AGENTCORE_LOG_GROUP = os.environ.get("AGENTCORE_LOG_GROUP", "aws/spans")
LOGS_QUERY_DEADLINE_S = 120.0
TERMINAL_QUERY_STATUSES = {"Complete", "Failed", "Cancelled", "Timeout", "Unknown"}
# Logs Insights query from the AWS documentation page linked above.
LOGS_INSIGHTS_QUERY = """fields @timestamp, @message
| filter ispresent(scope.name) and ispresent(attributes.session.id)
| filter attributes.session.id = "{session_id}"
| sort @timestamp asc"""


def download_session_spans(
    session_id: str, log_group: str, region: str, deadline_s: float = LOGS_QUERY_DEADLINE_S
) -> list[dict[str, Any]]:
    import boto3

    logs = boto3.client("logs", region_name=region)
    end = dt.datetime.now(dt.UTC)
    start = end - dt.timedelta(minutes=60)
    query_id = logs.start_query(
        logGroupName=log_group,
        startTime=int(start.timestamp()),
        endTime=int(end.timestamp()),
        queryString=LOGS_INSIGHTS_QUERY.format(session_id=session_id),
    )["queryId"]
    give_up = time.monotonic() + deadline_s
    while (result := logs.get_query_results(queryId=query_id))["status"] not in TERMINAL_QUERY_STATUSES:
        if time.monotonic() > give_up:
            logs.stop_query(queryId=query_id)
            raise TimeoutError(f"Logs Insights query still {result['status']} after {deadline_s:.0f}s")
        time.sleep(1)
    if result["status"] != "Complete":
        raise RuntimeError(f"Logs Insights query ended with status {result['status']}")
    return [
        json.loads(f["value"])
        for row in result["results"]
        for f in row
        if f["field"] == "@message" and f["value"].strip().startswith("{")
    ]


agentcore_missing = [
    name
    for name, ok in (
        ("STOCKROOM_MODE=live", config.is_live),
        ("STOCKROOM_CONFIRM_AWS_SPEND=1", config.confirm_aws_spend),
        ("AGENTCORE_SESSION_ID", bool(AGENTCORE_SESSION_ID)),
    )
    if not ok
]
if not agentcore_missing:
    import boto3

    session_spans = download_session_spans(AGENTCORE_SESSION_ID, AGENTCORE_LOG_GROUP, config.aws_region)
    agentcore = boto3.client("bedrock-agentcore", region_name=config.aws_region)
    response = agentcore.evaluate(
        evaluatorId="Builtin.Helpfulness", evaluationInput={"sessionSpans": session_spans}
    )
    for result in response["evaluationResults"]:
        print(result.get("evaluatorId"), result.get("value"), result.get("label"), result.get("errorCode"))
else:
    print("AgentCore Evaluations skipped; missing:", ", ".join(agentcore_missing), "\n")
    print("Logs Insights query:\n" + LOGS_INSIGHTS_QUERY.format(session_id="<session-id>"))
    print('\ncall shape: bedrock-agentcore.evaluate(evaluatorId="Builtin.Helpfulness", '
          'evaluationInput={"sessionSpans": [...]})')

## 9. Optional reading: how CI reaches AWS with GitHub OIDC

Two workflows live in `.github/workflows/`:

* **`agent_eval_ci.yml`** — the PR gate. Mock mode, no credentials, no network: lint, data
  validation, unit tests, the golden suite over the MCP server, the Promptfoo suite, these
  notebooks, then `check_thresholds.py`. It requests no AWS permissions at all.
* **`agent_eval_nightly.yml`** — live mode, report only, started by hand (`workflow_dispatch`; the
  repo keeps no cron so Bedrock spend happens only when someone asks for a run). It declares `permissions: id-token:
  write`, and `aws-actions/configure-aws-credentials` exchanges the job's GitHub OIDC token for
  short-lived credentials of an IAM role (`AWS_OIDC_ROLE_ARN`, a repository variable). The role's
  trust policy (`docs/aws/oidc_trust_policy.json`) only accepts tokens whose `sub` is this
  repository's `main` branch, and its permissions policy (`docs/aws/iam_policy.json`) allows
  invoking only the two configured models and one S3 prefix. No long-lived access key exists
  anywhere.

In [ ]:
def show(path: Path, needles: tuple[str, ...]) -> None:
    print(f"--- {path.relative_to(REPO_ROOT)}")
    for line in path.read_text(encoding="utf-8").splitlines():
        if any(n in line for n in needles):
            print(line.rstrip())
    print()


workflows = REPO_ROOT / ".github" / "workflows"
if workflows.exists():
    show(workflows / "agent_eval_ci.yml", ("permissions:", "contents:", "pull-requests:", "STOCKROOM_MODE"))
    show(
        workflows / "agent_eval_nightly.yml",
        ("permissions:", "id-token:", "configure-aws-credentials", "role-to-assume", "STOCKROOM_MODE", "--no-gate"),
    )
    print((REPO_ROOT / "docs" / "aws" / "oidc_trust_policy.json").read_text(encoding="utf-8"))
else:
    print("workflow files not found next to this checkout; see .github/workflows in the repository")

## Exercise checklist

One line per graded exercise. With `STOCKROOM_STRICT_EXERCISES=1` this cell fails unless every
exercise passed; `make notebooks` runs the solution notebooks that way.

In [ ]:
exercise_summary(["day4.ex1", "day4.ex2", "day4.ex3", "day4.ex4"])

## Wrap-up

* Offline evals gate PRs; live runs report run-to-run spread; online evaluators score real
  sessions. Each catches something the others cannot.
* Grow the golden set with a teacher, but validate labels against the data; the agent's result
  classifies an accepted case, it never decides whether the case is valid.
* A red-team case is only as good as the attack it would catch: assert on the trajectory and
  check the assertions fail when the attack succeeds.
* Floors are product decisions; noise is measured; the allowed regression must exceed the noise.
  Mock-mode zero variance is reproducibility, not reliability.
* Aggregates hide small categories. Per-category invariants, cost limits and a paired per-case
  comparison against a provenance-checked baseline let the gate see what the averages cannot.
* Bedrock model evaluation and AgentCore Evaluations are the managed counterparts of the judge
  and the trace evaluators you built; both are called only with explicit, auditable consent.